# Frame Camera Mission Planning

This notebook demonstrates how to plan aerial photography missions using
HyPlan's `FrameCamera` sensor class. Unlike line scanners (which build
images one line at a time), frame cameras capture full 2D images per frame.

We cover:

1. Defining a frame camera (sensor dimensions, focal length, resolution)
2. Field of view calculations
3. Ground sample distance and footprint vs. altitude
4. Altitude planning for a target GSD
5. Critical ground speed for along-track coverage
6. Comparing cameras with different focal lengths
7. Integration with flight lines and swath polygons

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import geopandas as gpd

from hyplan.frame_camera import FrameCamera
from hyplan.flight_line import FlightLine
from hyplan.units import ureg

## 1. Define a Frame Camera

A `FrameCamera` is defined by its physical sensor dimensions, focal length,
pixel resolution, frame rate, and f-number. These are typical parameters
for a medium-format aerial mapping camera.

In [ ]:
# Medium-format mapping camera
mapping_cam = FrameCamera(
    name="PhaseOne iXM-100",
    sensor_width=ureg.Quantity(53.4, "mm"),
    sensor_height=ureg.Quantity(40.0, "mm"),
    focal_length=ureg.Quantity(50.0, "mm"),
    resolution_x=11664,
    resolution_y=8750,
    frame_rate=ureg.Quantity(1.5, "Hz"),
    f_speed=4.0,
)

print(f"Camera: {mapping_cam.name}")
print(f"  Sensor:      {mapping_cam.sensor_width} x {mapping_cam.sensor_height}")
print(f"  Focal length: {mapping_cam.focal_length}")
print(f"  Resolution:  {mapping_cam.resolution_x} x {mapping_cam.resolution_y}")
print(f"  Frame rate:  {mapping_cam.frame_rate}")
print(f"  f-number:    f/{mapping_cam.f_speed}")
print(f"  FOV (H):     {mapping_cam.fov_x:.1f} deg")
print(f"  FOV (V):     {mapping_cam.fov_y:.1f} deg")

## 2. Ground Sample Distance and Footprint

GSD tells you the physical size of a pixel on the ground. The footprint
is the total area captured per frame. Both scale linearly with altitude.

In [ ]:
altitudes_m = np.arange(500, 5001, 250)

gsd_x_vals = []
gsd_y_vals = []
fp_width_vals = []
fp_height_vals = []

for alt in altitudes_m:
    alt_q = ureg.Quantity(alt, "meter")
    gsd = mapping_cam.ground_sample_distance(alt_q)
    fp = mapping_cam.footprint_at(alt_q)
    gsd_x_vals.append(gsd["x"].to(ureg.cm).magnitude)
    gsd_y_vals.append(gsd["y"].to(ureg.cm).magnitude)
    fp_width_vals.append(fp["width"].to(ureg.meter).magnitude)
    fp_height_vals.append(fp["height"].to(ureg.meter).magnitude)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(altitudes_m, gsd_x_vals, "o-", markersize=3, label="GSD x (across-track)")
axes[0].plot(altitudes_m, gsd_y_vals, "s-", markersize=3, label="GSD y (along-track)")
axes[0].set_xlabel("Altitude AGL (m)")
axes[0].set_ylabel("GSD (cm)")
axes[0].set_title("Ground Sample Distance vs. Altitude")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(altitudes_m, fp_width_vals, "o-", markersize=3, label="Width (across-track)")
axes[1].plot(altitudes_m, fp_height_vals, "s-", markersize=3, label="Height (along-track)")
axes[1].set_xlabel("Altitude AGL (m)")
axes[1].set_ylabel("Footprint (m)")
axes[1].set_title("Ground Footprint vs. Altitude")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Altitude for a Target GSD

Given a desired GSD, compute the required flight altitude. This is
the inverse of the GSD calculation.

In [ ]:
target_gsds_cm = [2, 3, 5, 10, 15, 20, 30, 50]

rows = []
for gsd_cm in target_gsds_cm:
    gsd_m = gsd_cm / 100.0
    gsd_q = ureg.Quantity(gsd_m, "meter")
    alt = mapping_cam.altitude_agl_for_ground_sample_distance(gsd_q, gsd_q)
    fp = mapping_cam.footprint_at(alt)
    rows.append({
        "Target GSD (cm)": gsd_cm,
        "Required Altitude (m)": f"{alt.to(ureg.meter).magnitude:.0f}",
        "Required Altitude (ft)": f"{alt.to(ureg.feet).magnitude:.0f}",
        "Footprint Width (m)": f"{fp['width'].to(ureg.meter).magnitude:.0f}",
        "Footprint Height (m)": f"{fp['height'].to(ureg.meter).magnitude:.0f}",
    })

alt_df = pd.DataFrame(rows)
alt_df

## 4. Critical Ground Speed

The critical ground speed is the maximum speed at which consecutive frames
still provide contiguous along-track coverage (no gaps). Exceeding this
speed requires higher frame rates or accepting gaps between frames.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

cgs_vals = [
    mapping_cam.critical_ground_speed(ureg.Quantity(a, "meter")).to(ureg.knot).magnitude
    for a in altitudes_m
]

ax.plot(altitudes_m, cgs_vals, "o-", markersize=4, color="steelblue")
ax.axhline(y=120, color="red", linestyle="--", alpha=0.5, label="B200 cruise (~120 kt low alt)")
ax.set_xlabel("Altitude AGL (m)")
ax.set_ylabel("Critical Ground Speed (knots)")
ax.set_title(f"{mapping_cam.name} — Max Speed for Contiguous Coverage")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Focal Length Comparison

Changing the focal length trades off between GSD (resolution) and footprint (coverage).
A longer focal length gives finer resolution but a narrower field of view.

In [ ]:
focal_lengths = [35, 50, 70, 100, 150]
cameras = {}

for fl in focal_lengths:
    cameras[f"{fl}mm"] = FrameCamera(
        name=f"Cam {fl}mm",
        sensor_width=ureg.Quantity(53.4, "mm"),
        sensor_height=ureg.Quantity(40.0, "mm"),
        focal_length=ureg.Quantity(fl, "mm"),
        resolution_x=11664,
        resolution_y=8750,
        frame_rate=ureg.Quantity(1.5, "Hz"),
        f_speed=4.0,
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, cam in cameras.items():
    gsd_vals = [
        cam.ground_sample_distance(ureg.Quantity(a, "meter"))["x"].to(ureg.cm).magnitude
        for a in altitudes_m
    ]
    fp_vals = [
        cam.footprint_at(ureg.Quantity(a, "meter"))["width"].to(ureg.meter).magnitude
        for a in altitudes_m
    ]
    axes[0].plot(altitudes_m, gsd_vals, "o-", markersize=3, label=label)
    axes[1].plot(altitudes_m, fp_vals, "s-", markersize=3, label=label)

axes[0].set_xlabel("Altitude AGL (m)")
axes[0].set_ylabel("GSD (cm)")
axes[0].set_title("GSD vs. Altitude by Focal Length")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Altitude AGL (m)")
axes[1].set_ylabel("Footprint Width (m)")
axes[1].set_title("Footprint Width vs. Altitude by Focal Length")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Footprint Visualization

Visualize the ground footprint of the camera at a specific altitude,
showing the aspect ratio of the captured frame.

In [ ]:
altitude = ureg.Quantity(2000, "meter")
fp = mapping_cam.footprint_at(altitude)
gsd = mapping_cam.ground_sample_distance(altitude)

w = fp["width"].to(ureg.meter).magnitude
h = fp["height"].to(ureg.meter).magnitude

fig, ax = plt.subplots(figsize=(8, 6))
rect = mpatches.Rectangle(
    (-w / 2, -h / 2), w, h,
    facecolor="lightblue", edgecolor="navy", linewidth=2, alpha=0.6,
)
ax.add_patch(rect)
ax.plot(0, 0, "r+", markersize=15, markeredgewidth=2, label="Nadir")

ax.set_xlim(-w * 0.7, w * 0.7)
ax.set_ylim(-h * 0.7, h * 0.7)
ax.set_aspect("equal")
ax.set_xlabel("Across-track (m)")
ax.set_ylabel("Along-track (m)")
ax.set_title(
    f"{mapping_cam.name} Footprint at {altitude}\n"
    f"{w:.0f} x {h:.0f} m — GSD: {gsd['x'].to(ureg.cm).magnitude:.1f} cm"
)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Flight Line Integration

Compute coverage for a frame camera along a flight line. Each frame
captures a discrete footprint; we can calculate how many frames are
captured along the line at a given speed.

In [ ]:
fl = FlightLine.center_length_azimuth(
    lat=34.4, lon=-119.8,
    length=ureg.Quantity(10, "km"),
    az=270.0,
    altitude_msl=ureg.Quantity(6500, "feet"),  # ~2000 m AGL
    site_name="Coastal Survey",
)

speed = ureg.Quantity(100, "knot")
altitude_agl = ureg.Quantity(2000, "meter")
fp = mapping_cam.footprint_at(altitude_agl)
gsd = mapping_cam.ground_sample_distance(altitude_agl)
cgs = mapping_cam.critical_ground_speed(altitude_agl)

# Number of frames along the line
line_length_m = fl.length.to(ureg.meter).magnitude
frame_period = (1 / mapping_cam.frame_rate).to(ureg.second).magnitude
speed_mps = speed.to(ureg.meter / ureg.second).magnitude
n_frames = int(line_length_m / (speed_mps * frame_period))
along_track_spacing = speed_mps * frame_period

print(f"Flight line: {fl.site_name}")
print(f"  Length:     {fl.length.to(ureg.km):.1f}")
print(f"  Speed:      {speed}")
print(f"  Alt AGL:    {altitude_agl}")
print(f"\nCamera performance:")
print(f"  GSD:                {gsd['x'].to(ureg.cm):.1f}")
print(f"  Footprint:          {fp['width'].to(ureg.meter):.0f} x {fp['height'].to(ureg.meter):.0f}")
print(f"  Critical speed:     {cgs.to(ureg.knot):.0f}")
print(f"  Frames captured:    {n_frames}")
print(f"  Along-track spacing: {along_track_spacing:.1f} m")
print(f"  Along-track overlap: {max(0, (1 - along_track_spacing / fp['height'].to(ureg.meter).magnitude)) * 100:.0f}%")

## Summary

| Feature | Method/Property | Purpose |
|---------|----------------|----------|
| Camera definition | `FrameCamera(name, sensor_width, ...)` | Define camera optics and sensor |
| Field of view | `fov_x`, `fov_y` | Horizontal and vertical FOV |
| Ground sample distance | `ground_sample_distance(altitude)` | Pixel size on ground (x, y) |
| Altitude planning | `altitude_agl_for_ground_sample_distance(gsd_x, gsd_y)` | Required altitude for target GSD |
| Footprint | `footprint_at(altitude)` | Frame dimensions on ground |
| Speed limit | `critical_ground_speed(altitude)` | Max speed for contiguous frames |
| Terrain corners | `footprint_corners()` (static) | Ray-terrain intersection for corners |